## Preparação do Ambiente e Carregamento dos Dados

### Instalação de Pacotes

In [1]:
! pip install --upgrade -q xgboost scikit-learn imbalanced-learn matplotlib optuna plotly nbformat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 15.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 80.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 68.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.8 which is incompatible.
bigframes 2.26.0 requi

### Importação de Bibliotecas

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import xgboost as xgb
from xgboost import XGBClassifier

import optuna
import pickle
import optuna.visualization as vis

#from utils.constants import *

### Constantes

In [3]:
CATEGORICAL_COLUMNS = {
    "gestante_paciente", "raca_cor_paciente", "sigla_uf_residencia"
}

# --- Training/Testing Constants ---

RANDOM_STATE = 42
TEST_RATIO = 0.15
N_FOLDS = 5
N_CLASSES = 3

TARGET_NAMES = ["low_risk", "alarm", "severe"]
TARGET_NAMES_COARSE = ["low_risk", "high_risk"]
TARGET_NAMES_FINE = ["alarm", "severe"]

TARGET_LABEL_MAP = {name: idx for idx, name in enumerate(TARGET_NAMES)}
LABEL_TARGET_MAP = {idx: name for idx, name in enumerate(TARGET_NAMES)}

### Carregamento dos Dados

In [4]:
df = pd.read_csv("../input/sinan-sbcas/dataset-processed-gb.csv")

categorical_features = list(CATEGORICAL_COLUMNS)

for col in categorical_features:
    df[col] = df[col].astype('category')

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

## Treinamento e Otimização do Modelo

### Treinamento com Validação Cruzada

In [ ]:
def train_on_folds(params):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        # XGBoost with device='cuda' will handle data transfer
        # No need for explicit cp.asarray here, as XGBoost can process pandas DFs directly
        # when enable_categorical=True and device='cuda'.
        # The data will implicitly be moved to GPU by XGBoost itself.

        model = XGBClassifier(**params)

        try:
            model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            sample_weight=sample_weights,
            verbose=False
            )
            preds = model.predict(X_valid)
            
            # If GPU was used for training, predictions are still on GPU (if applicable).
            # Need to convert y_valid to numpy for f1_score.
            # XGBoost's predict for DMatrix outputs numpy arrays by default if not on GPU, but with GPU it might be CuPy.
            # Ensure y_valid is numpy for comparison.
            y_valid_np = y_valid.to_numpy() if isinstance(y_valid, pd.Series) else y_valid

            f1 = f1_score(y_valid_np, preds, average='macro')
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0

    return np.mean(scores), np.std(scores)


### Definição da Função Objetivo

In [5]:
FIXED_PARAMS = {
    "n_estimators": 2000,
    "early_stopping_rounds": 5,
    "enable_categorical": True,
    "objective": "multi:softprob",
    "num_class": N_CLASSES,
    "tree_method": "hist",
    "device": "cuda",
    "eval_metric": "mlogloss",
    "random_state": RANDOM_STATE,
    "verbosity": 0
}

In [ ]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 10, 16),
        
        # Afunilando o Learning Rate ao redor do 0.09
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.15),
        
        # Testando amostragem alta, mas permitindo leve ruído
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        
        # Reduzindo o range de regularização agressiva
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 15),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 30),
        
        # Estabilizando o peso mínimo do filho
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 12),
        
        # Mantendo Gamma baixo para não travar o crescimento
        'gamma': trial.suggest_float('gamma', 0, 2.0),
        
        **FIXED_PARAMS
    }

    avg, _ = train_on_folds(params)
    
    return avg

### Otimização com Optuna

In [7]:
study = optuna.create_study(study_name="xgboost_study_cuda", direction="maximize")
study.enqueue_trial({'max_depth': 14, 'learning_rate': 0.091, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_alpha': 10, 'reg_lambda': 2, 'min_child_weight': 10, 'gamma': 0.0})
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

[I 2026-02-08 06:45:46,049] A new study created in memory with name: xgboost_study_cuda


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-02-08 06:46:49,476] Trial 1 finished with value: 0.5178337708673819 and parameters: {'max_depth': 15, 'learning_rate': 0.11800278972983767, 'subsample': 0.9700455871379918, 'colsample_bytree': 0.7212100919763235, 'reg_alpha': 9.148632735546776, 'reg_lambda': 26.913950757308175, 'min_child_weight': 7, 'gamma': 1.8129147725372208}. Best is trial 1 with value: 0.5178337708673819.
[I 2026-02-08 06:48:23,219] Trial 3 finished with value: 0.5169558624171297 and parameters: {'max_depth': 14, 'learning_rate': 0.05688083940625101, 'subsample': 0.8103789796067498, 'colsample_bytree': 0.7860859202699535, 'reg_alpha': 14.81290428813087, 'reg_lambda': 11.043592216854282, 'min_child_weight': 9, 'gamma': 1.5101922011850213}. Best is trial 1 with value: 0.5178337708673819.
[I 2026-02-08 06:50:15,346] Trial 5 finished with value: 0.5188649261343462 and parameters: {'max_depth': 13, 'learning_rate': 0.1101152768754336, 'subsample': 0.8218981612762469, 'colsample_bytree': 0.7815073591120547, 'reg

In [ ]:
output_dir = Path('opt-xgb')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

### Visualizações do Optuna

In [ ]:
display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

### Melhor Modelo

In [31]:
final_params = {**best_trial.params, **FIXED_PARAMS}

In [32]:
avg_f1_final, std_f1_final = train_on_folds(final_params)

In [33]:
print(avg_f1_final)
print(std_f1_final)

0.5568114616852097
0.0022681669137905413
